In [1]:
import os
import math
import re
from collections import Counter
import pandas as pd
import numpy as np
import torch
from tqdm.auto import tqdm
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models

load_dotenv()

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"Hardware accelerator: {device}")

Hardware accelerator: mps


In [2]:
# Connect to Qdrant Database
QDRANT_HOST = os.getenv("QDRANT_HOST", "localhost")
QDRANT_PORT = int(os.getenv("QDRANT_PORT", 6333))
QDRANT_URL = os.getenv("QDRANT_URL", f"http://{QDRANT_HOST}:{QDRANT_PORT}")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", None)

print(f"Attempting connection to Qdrant at: {QDRANT_URL}...")

client = None
if QDRANT_URL:
    try:
        remote_client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, check_compatibility=False, timeout=5)
        remote_client.get_collections()
        client = remote_client
        print(f"Successfully connected to Qdrant Vector Server at {QDRANT_URL}!")
    except Exception as e:
        print(f"Standalone Qdrant server not available at {QDRANT_URL} ({e}).")

if client is None:
    client = QdrantClient(":memory:")
    print("Connected to Qdrant In-Memory Session!")

Attempting connection to Qdrant at: https://67b23fca-0482-4a4e-9cdd-e579a9f6eced.europe-west3-0.gcp.cloud.qdrant.io...
Successfully connected to Qdrant Vector Server at https://67b23fca-0482-4a4e-9cdd-e579a9f6eced.europe-west3-0.gcp.cloud.qdrant.io!


In [3]:
# Initialize Dense Model (PubMedBERT)
DENSE_MODEL_NAME = "NeuML/pubmedbert-base-embeddings"
print(f"Loading Dense Embedding Model: {DENSE_MODEL_NAME}...")

try:
    dense_model = SentenceTransformer(DENSE_MODEL_NAME, device=device, local_files_only=True)
except Exception:
    dense_model = SentenceTransformer(DENSE_MODEL_NAME, device=device)

DENSE_VECTOR_SIZE = 768
print(f"Dense model initialized successfully. Embedding dimension: {DENSE_VECTOR_SIZE}")

Loading Dense Embedding Model: NeuML/pubmedbert-base-embeddings...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Dense model initialized successfully. Embedding dimension: 768


In [4]:
# Define BM25 Sparse Vector Encoder for Qdrant
class BM25Encoder:
    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.doc_freqs = Counter()
        self.num_docs = 0
        self.avgdl = 1.0

    def tokenize(self, text):
        return re.findall(r'\w+', str(text).lower())

    def _hash_token(self, token):
        # Hashing tokens to 32-bit positive integer index for Qdrant sparse vector
        return abs(hash(token)) % (2**31 - 1)

    def fit(self, texts):
        self.num_docs = len(texts)
        total_length = 0
        for text in texts:
            tokens = self.tokenize(text)
            total_length += len(tokens)
            for token in set(tokens):
                self.doc_freqs[token] += 1
        self.avgdl = total_length / self.num_docs if self.num_docs > 0 else 1.0

    def encode(self, text):
        tokens = self.tokenize(text)
        doc_len = len(tokens)
        counts = Counter(tokens)
        
        indices = []
        values = []
        for token, count in counts.items():
            df_val = self.doc_freqs.get(token, 1)
            # BM25 Inverse Document Frequency (IDF)
            idf = math.log((self.num_docs - df_val + 0.5) / (df_val + 0.5) + 1.0)
            # BM25 Term Frequency (TF)
            tf = (count * (self.k1 + 1)) / (count + self.k1 * (1 - self.b + self.b * (doc_len / self.avgdl)))
            weight = idf * tf
            if weight > 0:
                indices.append(self._hash_token(token))
                values.append(float(weight))
                
        return models.SparseVector(indices=indices, values=values)

print("Fitting BM25 Sparse Vectorizer on dataset corpus...")
sparse_encoder = BM25Encoder()
# sparse_encoder.fit(df["combined_text"].tolist())
print("BM25 Sparse Encoder fitted successfully!")

Fitting BM25 Sparse Vectorizer on dataset corpus...
BM25 Sparse Encoder fitted successfully!


In [5]:
COLLECTION_NAME = "medical_knowledge_base_hybrid"
def hybrid_search(query_text=None, query=None, top_k=3):
    q_text = query_text or query
    query_dense = dense_model.encode(q_text, normalize_embeddings=True).tolist()
    query_sparse = sparse_encoder.encode(q_text)
    
    res = client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=[
            models.Prefetch(query=query_dense, using="text-dense", limit=top_k * 3),
            models.Prefetch(query=query_sparse, using="text-sparse", limit=top_k * 3),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=top_k
    )
    return res.points

In [6]:
# Initialize Gemini Client (via OpenAI Compatibility Endpoint)
from openai import OpenAI
import json

gemini_api_key = os.getenv("GEMINI_API_KEY")
if not gemini_api_key:
    print("Warning: GEMINI_API_KEY not set in .env!")

openai_client = OpenAI(
    api_key=gemini_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)
print("Gemini 2.5 Flash client initialized successfully.")

Gemini 2.5 Flash client initialized successfully.


In [7]:
# System Instructions for Medical QA Agent
instructions = """
You are a knowledgeable and compassionate Medical QA Assistant.
Your task is to provide accurate, helpful, and evidence-based information for user medical questions.

Instructions:
1. Always use the 'search' function to search the Medical QA Knowledge Base for condition details, symptoms, causes, and treatments.
2. Formulate clear search queries using key medical terminology from the user's question.
3. Perform additional searches if necessary to gather complete context or explore related medical aspects.
4. Synthesize the search findings into a well-structured, user-friendly response.
5. Conclude your response by asking if the user has follow-up questions or other health topics to explore.
6. IMPORTANT: Always use native JSON tool calls. Do NOT output text tags like <function=...> in raw text responses.
""".strip()
print("Medical Assistant instructions defined with guardrails.")

Medical Assistant instructions defined with guardrails.


In [8]:
# Tool Execution Handler
def make_call(tool_call):
    args = json.loads(tool_call.function.arguments)
    
    if tool_call.function.name == "search":
        results = hybrid_search(**args)
        formatted_results = [
            {
                "score": round(r.score, 4),
                "focus_area": r.payload.get("focus_area"),
                "question": r.payload.get("question"),
                "answer": r.payload.get("answer"),
                "source": r.payload.get("source"),
            }
            for r in results
        ]
        output_json = json.dumps(formatted_results, indent=2)
    else:
        output_json = json.dumps({"error": "Unknown tool"})

    return {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": output_json,
    }

In [9]:
# Tool Definition for Groq Function Calling
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the Medical QA Knowledge Base for medical entries matching the query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query_text": {
                    "type": "string",
                    "description": "Medical search query text to look up in the knowledge base.",
                },
                "top_k": {
                    "type": "integer",
                    "description": "Number of relevant documents to retrieve (default 5).",
                }
            },
            "required": ["query_text"],
            "additionalProperties": False,
        },
    }
}

In [10]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [11]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["focus_area"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [12]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(question=question, context=context)
    return prompt.strip()

In [13]:
# Medical QA Agent Loop with Gemini 2.5 Flash
import re
from openai import BadRequestError

def parse_failed_generation(error_obj):
    try:
        body = getattr(error_obj, "body", {})
        failed_gen = body.get("failed_generation", "") if isinstance(body, dict) else str(error_obj)
        
        json_match = re.search(r"\{.*\}", failed_gen)
        if json_match:
            try:
                return json.loads(json_match.group(0))
            except Exception:
                pass
                
        query_match = re.search(r"\"query_text\":\s*\"([^\"]+)\"", failed_gen)
        if query_match:
            return {"query_text": query_match.group(1), "top_k": 5}
    except Exception:
        pass
    return None

def agent_loop(instructions, question, model="gemini-2.5-flash", max_turns=5) -> str:
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": question},
    ]

    for turn in range(1, max_turns + 1):
        print(f"\n=== Turn #{turn} ===")
        try:
            response = openai_client.chat.completions.create(
                model=model,
                messages=messages,
                tools=[search_tool],
                tool_choice="auto"
            )
            msg = response.choices[0].message
            
            # Build clean assistant message for OpenAI API compatibility
            assistant_msg = {"role": "assistant"}
            if msg.content:
                assistant_msg["content"] = msg.content
            if msg.tool_calls:
                assistant_msg["tool_calls"] = [
                    {
                        "id": tc.id,
                        "type": "function",
                        "function": {
                            "name": tc.function.name,
                            "arguments": tc.function.arguments
                        }
                    }
                    for tc in msg.tool_calls
                ]
            
            messages.append(assistant_msg)

            if msg.tool_calls:
                for tool_call in msg.tool_calls:
                    print(f"Tool Call: {tool_call.function.name}({tool_call.function.arguments})")
                    tool_response = make_call(tool_call)
                    messages.append(tool_response)
            else:
                print("\n=================== FINAL ASSISTANT RESPONSE ===================\n")
                print(msg.content)
                return msg.content

        except BadRequestError as e:
            print(f"Tool Parsing Warning: {e.message if hasattr(e, 'message') else e}")
            parsed_args = parse_failed_generation(e)
            if parsed_args:
                print(f"-> Automatically recovered tool arguments: {parsed_args}")
                recovered_id = f"recovered_call_{turn}"
                messages.append({
                    "role": "assistant",
                    "tool_calls": [{
                        "id": recovered_id,
                        "type": "function",
                        "function": {
                            "name": "search",
                            "arguments": json.dumps(parsed_args)
                        }
                    }]
                })
                dummy_call = type("ToolCall", (), {
                    "id": recovered_id,
                    "function": type("Func", (), {
                        "name": "search",
                        "arguments": json.dumps(parsed_args)
                    })()
                })()
                tool_response = make_call(dummy_call)
                messages.append(tool_response)
            else:
                raise e

    return messages[-1].get("content", "")

In [16]:
QUERY = "When did the world cup end"
agent_loop(instructions, QUERY)


=== Turn #1 ===

=================== FINAL ASSISTANT RESPONSE ===================

I am a Medical QA Assistant and can only provide information related to medical questions. I cannot answer questions about the World Cup.


'I am a Medical QA Assistant and can only provide information related to medical questions. I cannot answer questions about the World Cup.'